In [28]:
!pip install py_vncorenlp

In [29]:
import pandas as pd
import py_vncorenlp
import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, f1_score

# 1. Load dataset

In [30]:
def load_data(file_path):
    """Load sentences and sentiment labels from a text file

    Args:
        file_path (str): The path to the folder containing sentiment.txt and sents.txt.

    Returns:
        tuple: A tuple containing a list of sentences and a DataFrame of sentiment labels.
    """

    sents_file_path = file_path + '/sents.txt'
    sentiments_file_path = file_path + '/sentiments.txt'

    # Load sentences
    sents = []
    with open(sents_file_path, 'r', encoding='utf-8') as f:
        sents = f.readlines()

    sents = [s.strip() for s in sents if s.strip()]

    # Load sentiment labels
    df = pd.read_csv(sentiments_file_path, sep=',', header=None, names=['sentiment'])
    sentiments = df['sentiment'].astype(int)

    return sents, sentiments

In [31]:
train_path = '/content/data/train'
dev_path = '/content/data/dev'
test_path = '/content/data/test'

In [32]:
# Load the datasets
train_sents, train_sentiments = load_data(train_path)
dev_sents, dev_sentiments = load_data(dev_path)
test_sents, test_sentiments = load_data(test_path)

In [33]:
print(f"Loaded {len(train_sents)} training sentences, {len(dev_sents)} dev sentences, and {len(test_sents)} test sentences.")

Loaded 11426 training sentences, 1583 dev sentences, and 3166 test sentences.


# 2. Data preprocessing

## 2.1 Download VnCoreNLP
**Using word segmenter before feeding to PhoBert**

In [34]:
vncorenlp_dir='/content/'

In [35]:
# Download the VnCoreNLP model if not already downloaded
py_vncorenlp.download_model(save_dir=vncorenlp_dir)

VnCoreNLP model folder /content already exists! Please load VnCoreNLP from this folder!


In [36]:
# Load the word and sentence segmentation component
if not os.path.exists(vncorenlp_dir):
    os.makedirs(vncorenlp_dir, exist_ok=True)
    print("Downloading VnCoreNLP...")
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg"],
        save_dir=vncorenlp_dir,
    )
else:
    print("VnCoreNLP already downloaded.")
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
      annotators=["wseg"],
      save_dir=vncorenlp_dir,
    )

VnCoreNLP already downloaded.


In [37]:
# Test the segmenter on a sample sentence
text = "Ông Nguyễn Khắc Chúc  đang làm việc tại Đại học Quốc gia Hà Nội. Bà Lan, vợ ông Chúc, cũng làm việc tại đây."
segmented_sentence = rdrsegmenter.word_segment(text)
print(f"Segmented sentence: {segmented_sentence}")

Segmented sentence: ['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']


## 2.2 Preprocessing data

In [61]:
def preprocess_sent(sent):
    """Preprocess sent by segmenting them into words.

    Args:
        sent (string): A sentence to preprocess.

    Returns:
        preprocess_sent: A preprocessed sentences.
    """
    segmented = rdrsegmenter.word_segment(sent)
    preprocess_sent = ' '.join(segmented)
    return preprocess_sent

In [38]:
def preprocess_sentences(sentences):
    """Preprocess sentences by segmenting them into words.

    Args:
        sentences (list): A list of sentences to preprocess.

    Returns:
        list: A list of preprocessed sentences.
    """
    preprocessed_sentences = []
    for sentence in sentences:
        preprocessed_sentence = preprocess_sent(sentence)
        preprocessed_sentences.append(preprocessed_sentence)
    return preprocessed_sentences

In [39]:
# Preprocess the training, dev, and test sentences
train_sents = preprocess_sentences(train_sents)
dev_sents = preprocess_sentences(dev_sents)
test_sents = preprocess_sentences(test_sents)

In [40]:
# Create DataFrames for the datasets
train_df = pd.DataFrame({'sentence': train_sents, 'labels': train_sentiments})
dev_df = pd.DataFrame({'sentence': dev_sents, 'labels': dev_sentiments})
test_df = pd.DataFrame({'sentence': test_sents, 'labels': test_sentiments})

In [41]:
train_df.head()

,sentence,labels
0,slide giáo_trình đầy_đủ .,2
1,"nhiệt_tình giảng_dạy , gần_gũi với sinh_viên .",2
2,đi học đầy_đủ full điểm chuyên_cần .,0
3,chưa áp_dụng công_nghệ_thông_tin và các thiết_...,0
4,"thầy giảng bài hay , có nhiều bài_tập ví_dụ ng...",2


In [42]:
# Convert DataFrames to Hugging Face Datasets to suit the Trainer API
train_dataset = HFDataset.from_pandas(train_df)
dev_dataset = HFDataset.from_pandas(dev_df)
test_dataset = HFDataset.from_pandas(test_df)

# 3. Train model

## 3.1 Tokenize dataset

In [43]:
# Load the tokenizer for PhoBERT
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [44]:
def tokenize(batch):
    """Tokenize a batch of sentences.

    Args:
        batch (dict): A dictionary containing sentences to tokenize.

    Returns:
        dict: A dictionary with tokenized sentences.
    """
    return tokenizer(batch["sentence"], padding=True, truncation=True, max_length=128)

In [45]:
# Tokenize the dataset
train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/3166 [00:00<?, ? examples/s]

In [46]:
# Set the format for the datasets to be compatible with PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
dev_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

## 3.2 Load model PhoBert

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [48]:
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base",
    num_labels=3 # Assuming 3 sentiment classes: negative, neutral, positive
)
model.to(device)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

## 3.3 Train model

In [49]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
)

In [50]:
def compute_metrics(pred):
    """Compute accuracy and F1 score for the predictions.
    Args:
        pred (PredictionOutput): The predictions output from the model.
    Returns:
        dict: A dictionary containing accuracy and F1 score.
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}

In [51]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2406762165.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [52]:
# Train the model
print("Starting training...")
trainer.train()

Starting training...


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.241400,0.289372,0.947568,0.858301
2,0.247100,0.307029,0.951990,0.848484
3,0.129800,0.288143,0.955780,0.870678
4,0.093500,0.313635,0.953885,0.865916
5,0.047000,0.331730,0.952622,0.865163


TrainOutput(global_step=14285, training_loss=0.16772803749535775, metrics={'train_runtime': 1803.0142, 'train_samples_per_second': 31.686, 'train_steps_per_second': 7.923, 'total_flos': 3039828454594260.0, 'train_loss': 0.16772803749535775, 'epoch': 5.0})

In [57]:
# Save model và tokenizer
trainer.save_model("saved_model")
tokenizer.save_pretrained("saved_model")

('saved_model/tokenizer_config.json',
 'saved_model/special_tokens_map.json',
 'saved_model/vocab.txt',
 'saved_model/bpe.codes',
 'saved_model/added_tokens.json')

In [58]:
!zip -r saved_model.zip saved_model

  adding: saved_model/ (stored 0%)
  adding: saved_model/bpe.codes (deflated 59%)
  adding: saved_model/config.json (deflated 52%)
  adding: saved_model/model.safetensors (deflated 17%)
  adding: saved_model/vocab.txt (deflated 55%)
  adding: saved_model/training_args.bin (deflated 52%)
  adding: saved_model/added_tokens.json (stored 0%)
  adding: saved_model/tokenizer_config.json (deflated 77%)
  adding: saved_model/special_tokens_map.json (deflated 57%)


# 4. Evaluate

## 4.1 Evaluate with test_dataset

In [55]:
trainer.evaluate(test_dataset)

{'eval_loss': 0.40058720111846924,
 'eval_accuracy': 0.9409349336702464,
 'eval_f1': 0.8387790724695607,
 'eval_runtime': 15.8208,
 'eval_samples_per_second': 200.117,
 'eval_steps_per_second': 50.061,
 'epoch': 5.0}

## 4.2 Test with new text

In [68]:
text = "Bài giảng hay!"
label_class = ['negative', 'neutral', 'positive']

In [63]:
preprocessed_text = preprocess_sent(text)
tokenize_text = tokenizer(preprocessed_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
prediction = model(**tokenize_text)

In [69]:
predicted_class = prediction.logits.argmax(dim=-1).item()

print("Predicted class:", label_class[predicted_class])

Predicted class: positive


# 5. Load saved model

In [79]:
# Load tokenizer and model
saved_tokenizer = AutoTokenizer.from_pretrained("saved_model")
saved_model = AutoModelForSequenceClassification.from_pretrained("saved_model")
saved_model.to(device)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [75]:
def preprocess_tokenize_sentence(sent):
    """Preprocess and tokenize sent to fix with phoBert input.

    Args:
        sent (string): A sentence to preprocess and tokenzie.

    Returns:
        tokenize_sent: A preprocessed and tokenzied sentences.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sent = sent.strip()
    segmented = rdrsegmenter.word_segment(sent)
    preprocess_sent = ' '.join(segmented)
    tokenize_sent = saved_tokenizer(preprocess_sent, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    return tokenize_sent


In [76]:
def predict_sentiment(sent):
  """
  Predict a sentence with saved model

  Args:
    sent (string): A sentence to predict

  Returns:
    predicted_class (string): A predicted class
  """

  tokenize_sent = preprocess_tokenize_sentence(sent)
  prediction = saved_model(**tokenize_sent)
  predicted_class = prediction.logits.argmax(dim=-1).item()
  return label_class[predicted_class]

In [77]:
sent = 'Hôm nay là thứ 7!'

In [80]:
result = predict_sentiment(sent)
print(result)

neutral
